In [ ]:
import os
from typing import List
from dotenv import load_dotenv
from langchain_openai  import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import END, StateGraph
from typing_extensions import TypedDict

load_dotenv()

#shared dict passed between nodes

class State(TypedDict):
    question: str 
    chat_history : List
    context : str
    answer : str


In [5]:
# Node1
embaddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

vector_store = Chroma(
    collection_name = "my_docs",
    embedding_function=embaddings,
    persist_directory="./chroma_db"
)

def retrive(state:State):
    question = state["question"]
    docs = vector_store.similarity_search(question,k=3)

    context_parts = []
    for i, doc in enumerate(doc,1):
        page = doc.metadata.get("page_number","?")
        context_parts.append(
            f"[{i}] Page {page} :\n {doc.page_content}"
        )
    context = "\n\n--\n\n".join(context_parts)

    print(f"Retrived context : {context}")

    return{
        **state,
        "context":context
    }

C:\Users\Rahul\AppData\Local\Temp\ipykernel_27604\3436727577.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [6]:
#node 2 : build prompt and call chat gpt 
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

def generate(state:State):
    question = state["question"]
    context = state["context"]
    chat_history = state["chat_history"]

    history_str = " "
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            history_str+= f"Human : {msg.content}\n"
        elif isinstance(msg, AIMessage):
            history_str += f"AI: {msg.content}"
    
    prompt = f"""
    You are a document assistant
    Answer only from the context below
    If asnwer not found then say "No context found"
    Context : {context}
    Chat_History  : {chat_history}
    Question : {question}
    Answer :
    """

    response = llm.invoke(prompt)
    answer = response.content

    updated_history = chat_history + [
        HumanMessage(content=question),
        AIMessage(content=answer)
    ]
    return{
        **state,
        "answer":answer,
        "chat_history" : updated_history
    }